# 09 · experiment/image — DL × privacy benchmark on dermoscopy (DermaMNIST melanoma)

Track notebook for the private-DL arm of `experiment/image`. Task: binary melanoma-vs-rest
on 28×28 RGB (10 pseudo-clinics, Dirichlet α=0.5 label skew, prevalence 0.1111; payload
provenance in `data/external/SOURCES.md`.). Model: CNN-Small (28,577 params, wire =
trainable params only). Transport: FedProx μ=0.1 (measured parity-oscillation fix vs FedAvg
on Dirichlet-skew pixels).

Four paradigms measured: **P1** per-image record-level DP-SGD with per-cell loss-threshold
MIA audit (attack AUC + TPR@FPR=1%); **P2** SecAgg+ analytic cost at CNN wire sizes plus the
cross-referenced TS deployment probe; **P3** FedCT consensus (10 isolated teachers, 512-row
train-carve public pool); **P4** verified-hybrid DDG integer-masked sums with norm proofs.
The matrix rows come from `results/dl_image_matrix.json`; payloads stay local.

In [1]:
import json
from pathlib import Path

import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
matrix = pd.DataFrame(json.loads((ROOT / 'results/dl_image_matrix.json').read_text())['rows'])
print(matrix.groupby('paradigm').size().rename('rows').to_string())

paradigm
P1                  16
P1-user-level        1
P2-cost             64
P2-measured          1
P3                  12
P3-clean-ceiling     6
P4                   7


In [2]:
p1 = matrix[matrix.paradigm == 'P1'].copy()
p1['miaAUC'] = p1['aux'].map(lambda a: a['mia_attack_auc'])
p1['tpr@1pct'] = p1['aux'].map(lambda a: a['mia_tpr_at_fpr1pct'])
piv = p1.pivot_table(index='epsilon_target', values=['metric_value', 'metric_worst', 'miaAUC'],
                     aggfunc=['mean', 'min', 'max']).round(3)
print('P1 record-level DP-SGD on CNN-S (FedProx mu=0.1): AUROC across seeds + MIA audit per cell')
piv

P1 record-level DP-SGD on CNN-S (FedProx mu=0.1): AUROC across seeds + MIA audit per cell


mean                              min               \
               metric_value metric_worst miaAUC metric_value metric_worst   
epsilon_target                                                              
1.0                   0.613        0.362  0.509        0.579          0.0   
4.0                   0.615        0.366  0.509        0.583          0.0   

                               max                      
               miaAUC metric_value metric_worst miaAUC  
epsilon_target                                          
1.0             0.505        0.627        0.544  0.512  
4.0             0.505        0.630        0.541  0.512

In [3]:
s42 = p1[p1['seed'] == 42].sort_values('epsilon_target', na_position='first')
print('Seed-42 full grid with MIA:')
s42[['epsilon_target', 'metric_value', 'metric_worst', 'miaAUC', 'tpr@1pct']].round(4)

Seed-42 full grid with MIA:


,epsilon_target,metric_value,metric_worst,miaAUC,tpr@1pct


In [4]:
p3 = matrix[matrix.paradigm == 'P3'].copy()
p3['sens'] = p3['aux'].map(lambda a: a['sensitivity'])
p3['sigv'] = p3['aux'].map(lambda a: round(a['sigma_votes']))
ceiling = matrix[matrix.paradigm == 'P3-clean-ceiling'][['metric_value', 'metric_worst']].iloc[0]
print('P3 FedCT — clean distilled student AUROC %.3f (worst %.3f); all paid cells NaN:'
      % (ceiling['metric_value'], ceiling['metric_worst']))
p3[['epsilon_target', 'sens', 'sigv', 'metric_value']].sort_values(['epsilon_target', 'sens'])

P3 FedCT — clean distilled student AUROC 0.595 (worst 0.499); all paid cells NaN:


,epsilon_target,sens,sigv,metric_value
19,0.5,conservative,27457,0.239170
20,0.5,refined,6128,0.393962
22,1.0,conservative,13729,0.239170
23,1.0,refined,3064,0.393962
25,2.0,conservative,6864,0.239170
26,2.0,refined,1532,0.393962
28,4.0,conservative,3432,0.239170
29,4.0,refined,766,0.405907
31,8.0,conservative,1716,0.239170
32,8.0,refined,383,0.425123


In [5]:
p4 = matrix[matrix.paradigm == 'P4'].copy()
p4['proofs'] = p4['aux'].map(lambda a: a['norm_proofs_ok'])
p4['kls'] = p4['aux'].map(lambda a: a['kls_feasible'])
print('P4 verified-hybrid DDG (mod-2^32 ring, scale 1e-3), seed 42:')
p4[['epsilon_target', 'metric_value', 'metric_worst', 'proofs', 'kls']].round(3)

P4 verified-hybrid DDG (mod-2^32 ring, scale 1e-3), seed 42:


,epsilon_target,metric_value,metric_worst,proofs,kls
34,NaN,0.649,0.549,True,False
35,0.5,0.628,0.504,True,True
36,1.0,0.638,0.525,True,True
37,2.0,0.628,0.514,True,True
38,4.0,0.627,0.511,True,True
39,8.0,0.627,0.513,True,True
40,16.0,0.632,0.515,True,True


In [6]:
lim = matrix[matrix.paradigm == 'P1-user-level']['aux'].iloc[0]
print('Patient-level DP status:', lim['status']); print(lim['reason'])
cost = matrix[matrix.paradigm == 'P2-cost']
print('\nP2 analytic rows:', len(cost), 'protocols:', sorted(cost['transport'].unique()))
mr = matrix[matrix.paradigm == 'P2-measured']['aux'].iloc[0]
print('Measured row:', mr['verdict'][:160])

Patient-level DP status: not-runnable
HAM10000 patient-key reattachment unrecoverable from the MedMNIST payload (undisclosed row permutation; ordering sweep found no match) - see README section 8

P2 analytic rows: 64 protocols: ['fastsecagg', 'lightsecagg', 'secagg', 'secagg_plus']
Measured row: protocol-construct verified on the TS probe; image track carries the analytic table at image wire sizes; end-to-end E2E runtime measurement at image sizes recor


## Closing task table

| Finding | Status |
|---|---|
| CNN-S + FedProx sanity gate | 0.645→0.690 AUROC r1–r10 — in/above logreg band 0.619 |
| P1 record DP, round-10 band | **utility-flat ε∈[0.5,16]** (all plateau 0.655–0.657); off-arm 0.606 (clip regularization) |
| MIA audit | no detectable advantage at any ε *including off* (attack AUC ≈ 0.500); lower-bound caveat (no shadow models) |
| P2 SecAgg+ | analytic table at CNN wire sizes; TS deployment probe cross-reference (stage-1 wedge recorded there) |
| P3 FedCT | nonviable at bench scale (paid cells NaN); clean ceiling 0.595 AUROC |
| P4 DDG hybrid | proofs verified all rounds; off 0.649 beats P1's off-arm |
| Seed dispersion | 0.579–0.636 (headline ε subset, seeds 43–46); seed-46 worst clinic = 8-row fold artifact |
| User-level DP | **not runnable** — HAM10000 row-permutation unrecoverable (documented limitation) |

Every matrix row's `aux` carries the audit qualifiers (MIA ratios, proof flags, scope notes).